In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os

# 1. DB 연결 (파일이 생성됩니다)
db_path = 'ecg_management.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [2]:
# 2. 테이블 생성 (스키마 설계 반영)
cursor.executescript("""
-- 1. 환자 테이블
CREATE TABLE IF NOT EXISTS Patients (
    patient_id TEXT PRIMARY KEY,
    session_id TEXT,
    gender TEXT,
    age INTEGER,
    underlying_disease TEXT,
    ecg_duration REAL            -- 초 단위까지 반영
);

-- 2. 측정 세션 테이블
CREATE TABLE IF NOT EXISTS EcgSessions (
    session_id TEXT PRIMARY KEY,
    patient_id TEXT,
    sampling_rate INT,
    FOREIGN KEY (patient_id) REFERENCES Patients(patient_id)
);

-- 3. 심전도 신호 데이터 테이블 (파일 경로 방식)
CREATE TABLE IF NOT EXISTS EcgSignals (
    signal_id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT,
    lead_name TEXT,
    file_path TEXT,
    FOREIGN KEY (session_id) REFERENCES EcgSessions(session_id)
);

-- 4. 진단 결과 테이블
CREATE TABLE IF NOT EXISTS DiagnosisResults (
    diag_id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT,
    arrhythmia_type TEXT,
    severity_color TEXT,
    confidence_score REAL,
    FOREIGN KEY (session_id) REFERENCES EcgSessions(session_id)
);
""")


In [3]:
# ---------------------------------------------------------
# 3. 데이터 삽입 (CSV 기반)
# ---------------------------------------------------------

# (1) Patients 테이블 채우기 (성인/아동 정보 합치기)
adults = pd.read_csv('adults-subject-info.csv')
children = pd.read_csv('children-subject-info.csv')
patients_all = pd.concat([adults, children], ignore_index=True)

# 시간 데이터 전처리
# 'ecg_duration' 열을 ':' 기준으로 나누기 (expand=True는 각각 다른 열로 분리해줍니다)
time_split = patients_all['ecg_duration'].str.split(':', expand=True)

# 각 열을 숫자(float) 타입으로 변환
# time_split[0]은 시간(h), [1]은 분(m), [2]은 초(s)가 됩니다.
h = time_split[0].astype(float)
m = time_split[1].astype(float)
s = time_split[2].astype(float)

# 3. 전체 초(total_seconds) 계산
patients_all['ecg_duration_sec'] = h * 3600 + m * 60 + s


# 스키마 컬럼명에 맞춰 데이터프레임 정리 (파일 내 실제 컬럼명에 맞게 수정 필요)
# 예: 환자 ID 컬럼이 'ID'라면 'patient_id'로 변경
patients_to_db = patients_all[['subject_id', 'file_name', 'gender', 'age', 'diagnosis', 'ecg_duration_sec']].copy()
patients_to_db.columns = ['patient_id', 'session_id', 'gender', 'age', 'underlying_disease', 'ecg_duration']
patients_to_db.to_sql('Patients', conn, if_exists='append', index=False)

# (2) EcgSessions 테이블 채우기
# 'beats_metadata.csv' 등을 활용해 고유 세션 정보를 생성하여 삽입합니다.
# 여기서는 예시로 파일에서 정보를 읽어온다고 가정합니다.
EcgSessions_raw = pd.read_csv('beats_metadata.csv')

# session_id 생성 (변수명을 EcgSessions_raw로 통일)
EcgSessions_raw['session_id'] = EcgSessions_raw['patient_id'].astype(str) + "_" + EcgSessions_raw['peak_index'].astype(str)
EcgSessions_raw['sampling_rate'] = int(977) #확실하게 정수형으로 넣게 해줌

# DB 스키마에 정의된 컬럼만 정확히 선택 (patient_id 포함)
EcgSessions_to_db = EcgSessions_raw[['session_id', 'patient_id', 'sampling_rate']].copy()

# 필터링된 데이터프레임(EcgSessions_to_db)을 저장
EcgSessions_to_db.to_sql('EcgSessions', conn, if_exists='append', index=False)

118173

In [4]:
# (3) EcgSignals 테이블 (NPY 파일 경로 저장)
# 11만개 조각의 원본인 .npy 파일의 경로를 기록합니다.
signals_data = {
    'session_id': EcgSessions['session_id'].tolist(),
    'lead_name': ['II'] * len(EcgSessions),
    'file_path': [os.path.abspath('beats_signals.npy')] * len(EcgSessions)
}
pd.DataFrame(signals_data).to_sql('EcgSignals', conn, if_exists='append', index=False)

# (4) DiagnosisResults 테이블 (분석 결과 저장)
# 'beat_features_full.csv'에 있는 진단 결과나 위험도를 저장합니다.
features = pd.read_csv('beat_features_full.csv')
# 세션별로 대표 진단명을 추출하거나 요약하여 저장하는 로직 필요
# 아래는 예시 구조입니다.
diagnosis = pd.DataFrame({
    'session_id': EcgSessions['session_id'].tolist(),
    'arrhythmia_type': ['Normal'] * len(EcgSessions), # 예시
    'severity_color': ['Blue'] * len(EcgSessions),    # 예시
    'confidence_score': [0.95] * len(EcgSessions)
})
diagnosis.to_sql('DiagnosisResults', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print("✅ SQLite 데이터베이스 구축이 완료되었습니다!")

NameError: name 'EcgSessions' is not defined